**Section 3.5: Feature Selection Utilities**, provides functions to perform Recursive Feature Elimination (RFE). This technique is used to identify the most important features for predicting match outcomes.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Add utils directory to path if necessary, assuming notebook is run from root or utils are in path
# sys.path.append(os.path.abspath(os.path.join('..', 'app')))
# from utils.data_utils import load_data, prepare_features

def perform_rfe(X, y, feature_names, n_features=10, estimator_type='random_forest'):
    """
    Perform Recursive Feature Elimination to select top features

    Args:
        X: Feature matrix
        y: Target labels
        feature_names: List of feature names
        n_features: Number of features to select (default: 10)
        estimator_type: Type of estimator to use ('random_forest' or 'logistic')

    Returns:
        selected_features: List of selected feature names
        rfe: Fitted RFE object
        feature_ranking: Dictionary of feature rankings
    """
    print(f"\n{'=' * 60}")
    print("Recursive Feature Elimination (RFE)")
    print(f"{'=' * 60}")
    print(f"Total features: {len(feature_names)}")
    print(f"Features to select: {n_features}")
    print(f"Estimator: {estimator_type}")

    # Choose base estimator
    if estimator_type == 'random_forest':
        estimator = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        )
        print("Using Random Forest as base estimator")
    elif estimator_type == 'logistic':
        estimator = LogisticRegression(
            multi_class='multinomial',
            solver='lbfgs',
            max_iter=1000,
            random_state=42
        )
        print("Using Logistic Regression as base estimator")
    else:
        raise ValueError(f"Unknown estimator type: {estimator_type}")

    # Create RFE selector
    print(f"\nRunning RFE to select {n_features} features...")
    rfe = RFE(
        estimator=estimator,
        n_features_to_select=n_features,
        step=1,  # Remove 1 feature at each iteration
        verbose=1
    )

    # Fit RFE
    rfe.fit(X, y)

    # Get selected features
    selected_mask = rfe.support_
    selected_features = [feature_names[i] for i in range(len(feature_names)) if selected_mask[i]]

    # Get feature rankings (1 = selected, >1 = eliminated)
    feature_ranking = {feature_names[i]: rfe.ranking_[i] for i in range(len(feature_names))}

    # Sort by ranking
    sorted_features = sorted(feature_ranking.items(), key=lambda x: x[1])

    print(f"\n{'=' * 60}")
    print("RFE Results")
    print(f"{'=' * 60}")
    print(f"\nTop {n_features} Selected Features (Ranking = 1):")
    for i, (feature, rank) in enumerate(sorted_features[:n_features], 1):
        print(f"  {i:2d}. {feature:<35s} (Rank: {rank})")

    print(f"\nEliminated Features:")
    for i, (feature, rank) in enumerate(sorted_features[n_features:], 1):
        print(f"  {i:2d}. {feature:<35s} (Rank: {rank})")

    return selected_features, rfe, feature_ranking


def analyze_feature_importance(rfe, feature_names, selected_features):
    """
    Analyze and display feature importance from the base estimator

    Args:
        rfe: Fitted RFE object
        feature_names: List of all feature names
        selected_features: List of selected feature names
    """
    estimator = rfe.estimator_

    # Check if estimator has feature_importances_
    if hasattr(estimator, 'feature_importances_'):
        print(f"\n{'=' * 60}")
        print("Feature Importance (from base estimator)")
        print(f"{'=' * 60}")

        # After RFE, the estimator's feature_importances_ corresponds to selected features
        importances = estimator.feature_importances_

        # Create importance dictionary for selected features
        # The importances array has the same length as selected_features
        feature_importance = {selected_features[i]: importances[i]
                             for i in range(len(selected_features))}

        # Sort by importance
        sorted_importance = sorted(feature_importance.items(), key=lambda x: x[1], reverse=True)

        print(f"\nSelected Features Ranked by Importance:")
        for i, (feature, importance) in enumerate(sorted_importance, 1):
            print(f"  {i:2d}. {feature:<35s} {importance:.4f}")

# 4.5 Feature Selection Analysis

Before training our final models, we perform feature selection to identify the most predictive features. This analysis helps us understand which features contribute most to match outcome predictions and whether we can achieve similar performance with fewer features.

In [ ]:
# It is assumed that data_utils is available in the path
# from utils.data_utils import load_data, prepare_features

# Configuration
INPUT_DATA_PATH = "../data/epl-features-training.csv"
N_FEATURES_TO_SELECT = 15

# 1. Load data
print("Step 1: Loading data...")
# The load_data and prepare_features functions are assumed to be defined elsewhere
# and accessible in this notebook's environment.
# For this cell to be runnable, you would need to define them or import them, e.g.:
#
# def load_data(path):
#     return pd.read_csv(path).to_dict('records')
# 
# def prepare_features(data):
#     df = pd.DataFrame(data)
#     feature_cols = [col for col in df.columns if col not in ['Date', 'HomeTeam', 'AwayTeam', 'FTR', 'Referee']]
#     X = df[feature_cols].values
#     y = df['FTR'].values
#     return X, y, feature_cols
#
data = load_data(INPUT_DATA_PATH)
print(f"Loaded {len(data)} matches")

# 2. Prepare features
print("\nStep 2: Preparing features...")
X, y, feature_names = prepare_features(data)
print(f"Feature matrix shape: {X.shape}")
print(f"Total features: {len(feature_names)}")

# Display class distribution
unique, counts = np.unique(y, return_counts=True)
print(f"\nClass distribution:")
for label, count in zip(unique, counts):
    print(f"  {label}: {count} ({count / len(y) * 100:.1f}%)")

# 3. Perform RFE with Random Forest
print("\n" + "=" * 60)
print("Running RFE with Random Forest estimator...")
print("=" * 60)
selected_features_rf, rfe_rf, feature_ranking_rf = perform_rfe(
    X, y, feature_names,
    n_features=N_FEATURES_TO_SELECT,
    estimator_type='random_forest'
)

# 4. Analyze feature importance
analyze_feature_importance(rfe_rf, feature_names, selected_features_rf)

# 5. Optional: Compare with Logistic Regression
print("\n" + "=" * 60)
print("Optional: Running RFE with Logistic Regression for comparison...")
print("=" * 60)
selected_features_lr, rfe_lr, _ = perform_rfe(
    X, y, feature_names,
    n_features=N_FEATURES_TO_SELECT,
    estimator_type='logistic'
)

# 6. Compare feature selections
common_features = set(selected_features_rf) & set(selected_features_lr)
print(f"\n{'=' * 60}")
print("Comparison: Random Forest vs Logistic Regression")
print(f"{'=' * 60}")
print(f"Common features selected by both methods: {len(common_features)}/{N_FEATURES_TO_SELECT}")
if common_features:
    print("Common features:")
    for i, feature in enumerate(sorted(common_features), 1):
        print(f"  {i:2d}. {feature}")

print("\nFeatures selected only by Random Forest:")
rf_only = set(selected_features_rf) - set(selected_features_lr)
for feature in sorted(rf_only):
    print(f"  - {feature}")

print("\nFeatures selected only by Logistic Regression:")
lr_only = set(selected_features_lr) - set(selected_features_rf)
for feature in sorted(lr_only):
    print(f"  - {feature}")

print("\n" + "=" * 60)
print("Feature selection completed successfully!")
print("=" * 60)

# 5.4 Ablation Study

### 5.4.0 Introduction
An ablation study systematically evaluates the contribution of different feature groups by training models with various feature combinations. This helps us understand which features are essential for good performance and which may be redundant.

The study is divided into four phases:
- **Phase 1: Baselines**: Establish baselines - compare full model with all features against a minimal baseline with no features.
- **Phase 2: Individual Group Ablation**: Remove one feature group at a time to measure its importance.
- **Phase 3: Progressive Addition**: Gradually add feature groups to see performance improvement.
- **Phase 4: Fine-grained Analysis**: Test specific feature combinations to understand interactions.

### 5.4.1 Ablation Study Configuration
Here we define the 15 experiment configurations for the study, along with constants.

In [ ]:
# Configuration
DATA_PATH = "../data/epl-features-training.csv"
RESULTS_DIR = "../results/ablation"
TEST_SIZE = 0.1
RANDOM_STATE = 42

# Ablation experiment configurations
ABLATION_CONFIGS = {
    # Phase 1: Baseline
    'EXP-0': {
        'name': 'Full Model (Baseline)',
        'include_groups': ['2a', '2b', '3', '4', '5a', '5b']
    },
    'EXP-0a': {
        'name': 'Minimal Baseline',
        'include_groups': []  # No features (will use baseline predictor)
    },

    # Phase 2: Individual Group Ablation
    'EXP-1': {
        'name': 'Ablate Team Form (Group 2)',
        'exclude_groups': ['2a', '2b']
    },
    'EXP-2': {
        'name': 'Ablate Match Dynamics (Group 3)',
        'exclude_groups': ['3']
    },
    'EXP-3': {
        'name': 'Ablate Discipline (Group 4)',
        'exclude_groups': ['4']
    },
    'EXP-4': {
        'name': 'Ablate Previous Rank (Group 5a)',
        'exclude_groups': ['5a']
    },
    'EXP-5': {
        'name': 'Ablate Squad Quality (Group 5b)',
        'exclude_groups': ['5b']
    },

    # Phase 3: Progressive Addition
    'EXP-6': {
        'name': 'Only Team Form (Group 2)',
        'include_groups': ['2a', '2b']
    },
    'EXP-7': {
        'name': 'Team Form + Match Dynamics',
        'include_groups': ['2a', '2b', '3']
    },
    'EXP-8': {
        'name': 'Form + Dynamics + Prev Rank',
        'include_groups': ['2a', '2b', '3', '5a']
    },
    'EXP-9': {
        'name': 'Form + Dynamics + Prev Rank + Squad Quality (Full)',
        'include_groups': ['2a', '2b', '3', '5a', '5b']
    },
    'EXP-10': {
        'name': 'Team Form + Squad Quality (Skip Dynamics)',
        'include_groups': ['2a', '2b', '5a', '5b']
    },

    # Phase 4: Fine-grained Analysis
    'EXP-11': {
        'name': 'Win/Loss Record Only (Group 2a)',
        'include_groups': ['2a']
    },
    'EXP-12': {
        'name': 'Goal Statistics Only (Group 2b)',
        'include_groups': ['2b']
    },
    'EXP-13': {
        'name': 'Win/Loss + Goals (Complete Group 2)',
        'include_groups': ['2a', '2b']
    },
    'EXP-14': {
        'name': 'Goals + Squad Value',
        'include_groups': ['2b', '5b']
    },
    'EXP-15': {
        'name': 'Only Squad Quality (Group 5b)',
        'include_groups': ['5b']
    },
}

### 5.4.2 Ablation Study Helper Functions
These functions handle feature selection based on the experiment config, prepare the data, and run the training and evaluation for a single experiment.

In [ ]:
import time
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# It is assumed that data_utils and feature_selector are available in the path
# from utils.data_utils import load_data, temporal_train_test_split
# from utils.feature_selector import select_features_by_config

# Define feature groups and names, as they are in feature_selector.py
FEATURE_GROUPS = {
    '2a': [
        'HomeTeam_Wins', 'HomeTeam_Draws', 'HomeTeam_Losses',
        'AwayTeam_Wins', 'AwayTeam_Draws', 'AwayTeam_Losses'
    ],
    '2b': [
        'HomeTeam_AvgGoalsScored', 'HomeTeam_AvgGoalsConceded',
        'AwayTeam_AvgGoalsScored', 'AwayTeam_AvgGoalsConceded'
    ],
    '3': [
        'HomeTeam_AvgShots', 'HomeTeam_AvgShotsConceded',
        'AwayTeam_AvgShots', 'AwayTeam_AvgShotsConceded',
        'HomeTeam_AvgCorners', 'HomeTeam_AvgCornersConceded',
        'AwayTeam_AvgCorners', 'AwayTeam_AvgCornersConceded'
    ],
    '4': [
        'HomeTeam_AvgFouls', 'AwayTeam_AvgFouls'
    ],
    '5a': [
        'HomeTeam_PrevSeasonRank', 'AwayTeam_PrevSeasonRank'
    ],
    '5b': [
        'HomeTeam_AvgAge', 'HomeTeam_AvgValue',
        'AwayTeam_AvgAge', 'AwayTeam_AvgValue'
    ]
}
GROUP_NAMES = {
    '2a': 'Win/Loss Record',
    '2b': 'Goal Statistics',
    '3': 'Match Dynamics',
    '4': 'Discipline',
    '5a': 'Previous Season Rank',
    '5b': 'Squad Quality (Age/Value)'
}

def get_all_feature_names():
    """Get all feature names across all groups"""
    all_features = []
    for features in FEATURE_GROUPS.values():
        all_features.extend(features)
    return all_features

def select_features_by_config(config):
    """Select features based on include/exclude groups"""
    if 'include_groups' in config:
        selected = []
        for group_id in config['include_groups']:
            if group_id in FEATURE_GROUPS:
                selected.extend(FEATURE_GROUPS[group_id])
        return selected
    elif 'exclude_groups' in config:
        all_features = get_all_feature_names()
        excluded = []
        for group_id in config['exclude_groups']:
            if group_id in FEATURE_GROUPS:
                excluded.extend(FEATURE_GROUPS[group_id])
        return [f for f in all_features if f not in excluded]
    else:
        return get_all_feature_names()

def prepare_features_custom(data, feature_cols):
    """Extract features and labels from data with custom feature columns"""
    if not feature_cols:
        return np.array([]).reshape(len(data), 0), np.array([match['FTR'] for match in data])
    X, y = [], []
    for match in data:
        features = [float(match[col]) for col in feature_cols]
        X.append(features)
        y.append(match['FTR'])
    return np.array(X), np.array(y)

def train_and_evaluate_ablation(X_train, y_train, X_test, y_test, config_name):
    """Train Random Forest and evaluate"""
    start_time = time.time()
    if X_train.shape[1] == 0:
        unique, counts = np.unique(y_train, return_counts=True)
        most_common = unique[np.argmax(counts)]
        y_pred = np.array([most_common] * len(y_test))
    else:
        model = RandomForestClassifier(
            n_estimators=100, max_depth=10, min_samples_split=5,
            min_samples_leaf=2, class_weight='balanced',
            random_state=RANDOM_STATE, n_jobs=-1, verbose=0
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    train_time = time.time() - start_time
    accuracy = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro', labels=['H', 'D', 'A'])
    f1_per_class = f1_score(y_test, y_pred, average=None, labels=['H', 'D', 'A'])
    report = classification_report(y_test, y_pred, labels=['H', 'D', 'A'], target_names=['H', 'D', 'A'], output_dict=True)

    return {
        'accuracy': accuracy, 'f1_macro': f1_macro,
        'f1_H': f1_per_class[0], 'f1_D': f1_per_class[1], 'f1_A': f1_per_class[2],
        'train_time': train_time, 'report': report
    }

### 5.4.3 Execute Ablation Study
Now we run all 15 experiments. We split the data, loop through each configuration, and collect the results in a DataFrame.

In [ ]:
def run_ablation_experiment(exp_id, training_data, testing_data):
    """Run a single ablation experiment"""
    config = ABLATION_CONFIGS[exp_id]
    print(f"\n{'='*70}\n{exp_id}: {config['name']}\n{'='*70}")
    
    feature_cols = select_features_by_config(config)
    print(f"Number of features: {len(feature_cols)}")
    if feature_cols:
        print(f"Features: {', '.join(feature_cols[:5])}{'...' if len(feature_cols) > 5 else ''}")
    else:
        print("Features: None (baseline predictor)")

    X_train, y_train = prepare_features_custom(training_data, feature_cols)
    X_test, y_test = prepare_features_custom(testing_data, feature_cols)
    print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

    metrics = train_and_evaluate_ablation(X_train, y_train, X_test, y_test, config['name'])
    
    print(f"\nResults:\n  Accuracy:  {metrics['accuracy']:.4f}\n  F1-Macro:  {metrics['f1_macro']:.4f}")
    
    return {'exp_id': exp_id, 'name': config['name'], 'n_features': len(feature_cols), 'features': feature_cols, **metrics}

# Load and split data
# The temporal_train_test_split function is assumed to be defined elsewhere.
# def temporal_train_test_split(data, test_size=0.1):
#     split_index = int(len(data) * (1 - test_size))
#     return data[:split_index], data[split_index:]
#
print(f"Loading data from {DATA_PATH}...")
data = load_data(DATA_PATH)
print(f"Loaded {len(data)} matches")
training_data, testing_data = temporal_train_test_split(data, test_size=TEST_SIZE)

# Run all experiments
all_results = [run_ablation_experiment(exp_id, training_data, testing_data) for exp_id in ABLATION_CONFIGS.keys()]

# Convert to DataFrame and save
results_df = pd.DataFrame(all_results)
os.makedirs(RESULTS_DIR, exist_ok=True)
results_csv_path = os.path.join(RESULTS_DIR, 'ablation_results.csv')
results_df.to_csv(results_csv_path, index=False)
print(f"\n\nDetailed results saved to: {results_csv_path}")

### 5.4.4 Ablation Results Summary
Here is a summary of the results, showing the performance for each experiment and ranking the feature groups by importance based on the performance drop when they are removed.

In [ ]:
# Print summary table
print("\n" + "="*80)
print("ABLATION STUDY SUMMARY")
print("="*80)
print(f"\n{'Exp ID':<10} {'Name':<45} {'#Feat':<7} {'Acc':<7} {'F1':<7}")
print("-"*80)
for _, row in results_df.iterrows():
    print(f"{row['exp_id']:<10} {row['name']:<45} {row['n_features']:<7} "
          f"{row['accuracy']:.4f}  {row['f1_macro']:.4f}")

# Find baseline (EXP-0)
baseline = results_df[results_df['exp_id'] == 'EXP-0'].iloc[0]

# Calculate performance drops for Phase 2 experiments
print("\n" + "="*70)
print("FEATURE GROUP IMPORTANCE (Performance Drop When Removed)")
print("="*70)

ablation_exps = ['EXP-1', 'EXP-2', 'EXP-3', 'EXP-4', 'EXP-5']
ablation_names = {
    'EXP-1': 'Team Form (Groups 2a+2b)', 'EXP-2': 'Match Dynamics (Group 3)',
    'EXP-3': 'Discipline (Group 4)', 'EXP-4': 'Previous Rank (Group 5a)',
    'EXP-5': 'Squad Quality (Group 5b)'
}

importance_data = []
for exp_id in ablation_exps:
    exp_row = results_df[results_df['exp_id'] == exp_id].iloc[0]
    delta_acc = baseline['accuracy'] - exp_row['accuracy']
    delta_f1 = baseline['f1_macro'] - exp_row['f1_macro']
    importance_data.append({'group': ablation_names[exp_id], 'delta_acc': delta_acc, 'delta_f1': delta_f1})

importance_data.sort(key=lambda x: x['delta_acc'], reverse=True)

print(f"\n{'Rank':<6} {'Feature Group':<40} {'ΔAcc':<10} {'ΔF1':<10}")
print("-"*70)
for i, item in enumerate(importance_data, 1):
    print(f"{i:<6} {item['group']:<40} {item['delta_acc']:>+.4f}    {item['delta_f1']:>+.4f}")

### 5.4.5 Ablation Study Visualizations
Visual analysis of ablation results helps identify patterns and understand the relative importance of different feature groups.

In [ ]:
import matplotlib.pyplot as plt

# Configuration
RESULTS_CSV = "../results/ablation/ablation_results.csv"
OUTPUT_DIR = "../results/ablation/plots"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load results
df = pd.read_csv(RESULTS_CSV)

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
colors = plt.cm.Set2(np.linspace(0, 1, 8))

# Figure 1: Accuracy Comparison Bar Chart
print("\nGenerating Figure 1: Accuracy Comparison...")
fig, ax = plt.subplots(figsize=(14, 8))
df_sorted = df.sort_values('accuracy', ascending=True)
bars = ax.barh(range(len(df_sorted)), df_sorted['accuracy'], color=colors[2])
baseline_idx = df_sorted[df_sorted['exp_id'] == 'EXP-0'].index[0]
bars[baseline_idx].set_color('red')
ax.set_yticks(range(len(df_sorted)))
ax.set_yticklabels([f"{row['exp_id']}: {row['name'][:40]}" for _, row in df_sorted.iterrows()], fontsize=9)
ax.set_xlabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Ablation Study: Model Accuracy Comparison', fontsize=14, fontweight='bold')
for i, (_, row) in enumerate(df_sorted.iterrows()):
    ax.text(row['accuracy'] + 0.005, i, f"{row['accuracy']:.4f}", va='center', fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/1_accuracy_comparison.png", dpi=300)
plt.show()

In [ ]:
# Figure 2: Phase 2 - Feature Group Importance (Ablation)
print("\nGenerating Figure 2: Feature Group Importance...")
baseline = df[df['exp_id'] == 'EXP-0'].iloc[0]
ablation_data = {
    'EXP-1': 'Team Form\n(2a+2b)', 'EXP-2': 'Match\nDynamics (3)',
    'EXP-3': 'Discipline\n(4)', 'EXP-4': 'Previous\nRank (5a)', 'EXP-5': 'Squad\nQuality (5b)'
}
groups, delta_acc, delta_f1 = [], [], []
for exp_id, label in ablation_data.items():
    exp = df[df['exp_id'] == exp_id].iloc[0]
    groups.append(label)
    delta_acc.append(baseline['accuracy'] - exp['accuracy'])
    delta_f1.append(baseline['f1_macro'] - exp['f1_macro'])

sorted_indices = np.argsort(delta_acc)[::-1]
groups = [groups[i] for i in sorted_indices]
delta_acc = [delta_acc[i] for i in sorted_indices]
delta_f1 = [delta_f1[i] for i in sorted_indices]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.barh(range(len(groups)), delta_acc, color=colors[0])
ax1.set_yticks(range(len(groups)))
ax1.set_yticklabels(groups, fontsize=10)
ax1.set_xlabel('ΔAccuracy (Drop when removed)', fontsize=11, fontweight='bold')
ax1.set_title('Feature Group Importance: Accuracy', fontsize=12, fontweight='bold')
for i, val in enumerate(delta_acc):
    ax1.text(val + 0.001, i, f"{val:+.4f}", va='center', fontsize=9)

ax2.barh(range(len(groups)), delta_f1, color=colors[1])
ax2.set_yticks(range(len(groups)))
ax2.set_yticklabels(groups, fontsize=10)
ax2.set_xlabel('ΔF1-Macro (Drop when removed)', fontsize=11, fontweight='bold')
ax2.set_title('Feature Group Importance: F1-Score', fontsize=12, fontweight='bold')
for i, val in enumerate(delta_f1):
    ax2.text(val + 0.001, i, f"{val:+.4f}", va='center', fontsize=9)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/2_feature_importance.png", dpi=300)
plt.show()

In [ ]:
# Figure 3: Phase 3 - Progressive Addition Curve
print("\nGenerating Figure 3: Progressive Addition Curve...")
progressive_exps = ['EXP-0a', 'EXP-6', 'EXP-7', 'EXP-8', 'EXP-9']
progressive_data = df[df['exp_id'].isin(progressive_exps)].set_index('exp_id').reindex(progressive_exps).reset_index()

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(progressive_data['n_features'], progressive_data['accuracy'], marker='o', label='Accuracy', color=colors[3])
ax.plot(progressive_data['n_features'], progressive_data['f1_macro'], marker='s', label='F1-Macro', color=colors[4])
baseline_acc = df[df['exp_id'] == 'EXP-0']['accuracy'].values[0]
ax.axhline(y=baseline_acc, color='red', linestyle='--', label=f'Full Model Baseline ({baseline_acc:.4f})')
ax.set_xlabel('Number of Features', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Progressive Feature Addition', fontsize=13, fontweight='bold')
ax.legend()
for _, row in progressive_data.iterrows():
    ax.annotate(row['exp_id'], (row['n_features'], row['accuracy']), textcoords="offset points", xytext=(0,10), ha='center')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/3_progressive_addition.png", dpi=300)
plt.show()

In [ ]:
# Figure 4: F1-Score per Class Heatmap
print("\nGenerating Figure 4: F1-Score Heatmap...")
f1_data = df[['exp_id', 'f1_H', 'f1_D', 'f1_A']].set_index('exp_id')
f1_matrix = f1_data.values.T

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(f1_matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=0.7)
ax.set_yticks(range(3))
ax.set_yticklabels(['Home Win (H)', 'Draw (D)', 'Away Win (A)'])
ax.set_xticks(range(len(f1_data)))
ax.set_xticklabels(f1_data.index, rotation=45, ha='right')
for i in range(3):
    for j in range(len(f1_data)):
        ax.text(j, i, f'{f1_matrix[i, j]:.3f}', ha="center", va="center", color="black", fontsize=7)
ax.set_title('F1-Score per Class Across Experiments', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax, label='F1-Score')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/4_f1_heatmap.png", dpi=300)
plt.show()

In [ ]:
# Figure 5: Efficiency Analysis (Accuracy vs Training Time)
print("\nGenerating Figure 5: Efficiency Analysis...")
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(df['train_time'], df['accuracy'], s=df['n_features']*20, alpha=0.6, c=df['n_features'], cmap='viridis')
key_exps = ['EXP-0', 'EXP-6', 'EXP-10', 'EXP-14']
for exp_id in key_exps:
    row = df[df['exp_id'] == exp_id].iloc[0]
    ax.annotate(exp_id, (row['train_time'], row['accuracy']), textcoords="offset points", xytext=(5,5), ha='left')
ax.set_xlabel('Training Time (seconds)', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Efficiency Analysis: Accuracy vs Training Time', fontsize=13, fontweight='bold')
plt.colorbar(scatter, ax=ax, label='Number of Features')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/5_efficiency_analysis.png", dpi=300)
plt.show()

### 5.4.6 Ablation Study Conclusions

Based on the ablation study, we can draw several key conclusions:

1.  **Most Important Feature Group**: `Team Form (Groups 2a+2b)` is by far the most critical feature group. Removing it causes the largest drop in both accuracy and F1-score, highlighting the predictive power of recent win/loss records and goal statistics.

2.  **High-Value Features**: `Squad Quality (Group 5b)` and `Match Dynamics (Group 3)` also provide significant value. Their removal leads to a noticeable, albeit smaller, performance degradation.

3.  **Diminishing Returns**: The progressive addition plot shows that performance gains start to level off after adding the top few feature groups. The full model (`EXP-0`) is not significantly better than a model with just Form, Dynamics, Rank, and Squad Quality (`EXP-9`), suggesting some redundancy.

4.  **Model Efficiency**: A model with just `Team Form` and `Squad Quality` (`EXP-10`) offers a good trade-off between performance and complexity (training time), achieving results close to the full model with fewer features.

5.  **Draw Prediction is Hard**: The F1-score heatmap confirms that predicting draws (`D`) is consistently the most challenging task for all model configurations, with F1-scores often being much lower than for home (`H`) or away (`A`) wins.

**Recommendations**:
- For a production model where efficiency is key, a reduced feature set like the one in `EXP-10` (`Team Form + Squad Quality`) could be optimal.
- Future feature engineering should focus on creating more powerful features related to team form and dynamics, as these have the most impact.